In [0]:
%run ../config_setup

In [0]:
# Databricks Notebook: sim_06_api (parameterized)
# Schedule: Every 30 min (API polling simulation)

import uuid, random, datetime as dt
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
ROOT = LANDING
SEED = f"{ROOT}/_seed"

try:
    sites     = [r.asDict() for r in spark.read.json(f"{SEED}/sites").collect()]
    geofences = [r.asDict() for r in spark.read.json(f"{SEED}/geofences").collect()]
    if not sites: raise Exception()
    print(f"✅ Seed: {len(sites)} sites")
except:
    dbutils.notebook.exit("❌ SEED MISSING — run sim_00_seed first")

site_ids = [s["site_id"] for s in sites]
run_ts = dt.datetime.utcnow()
run_id = f"{run_ts:%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:6]}"

# ---- weather_snapshot (FK: site_id) ----
weather = []
for sid in random.sample(site_ids, k=min(5, len(site_ids))):
    weather.append({
        "weather_id": f"WX-{run_ts:%Y%m%d%H%M}-{sid}",
        "site_id": sid,
        "obs_ts": run_ts.isoformat()+"Z",
        "temp_c": round(random.uniform(15, 40), 1),
        "humidity": random.randint(30, 90),
        "wind_kph": round(random.uniform(0, 30), 1),
        "condition": random.choice(["clear","cloudy","rain","storm"]),
    })
spark.createDataFrame(weather).write.mode("append").json(
    f"{ROOT}/api/weather_snapshot/{run_id}.json")
print(f"  ✓ weather_snapshot: {len(weather)}")

# ---- geofence_lookup (FK: site_id, geofence_id) — only if not present ----
try:
    spark.read.json(f"{ROOT}/api/geofence_lookup").count()
    print("  ⏭  geofence_lookup exists — skipping")
except:
    gl = [{
        "geofence_id": g["geofence_id"],
        "site_id": g["site_id"],
        "boundary_geojson": '{"type":"Polygon","coordinates":[[[0,0],[1,0],[1,1],[0,1],[0,0]]]}',
        "updated_ts": run_ts.isoformat()+"Z",
    } for g in geofences]
    spark.createDataFrame(gl).write.mode("overwrite").json(
        f"{ROOT}/api/geofence_lookup/")
    print(f"  ✓ geofence_lookup: initial load ({len(gl)})")

print(f"\n✅ API run complete: {run_id}")